## Tiền xử lý Dữ liệu Tiếng Việt (Deep Dive)

Trong tiếng Việt, đặc điểm khác biệt nhất so với tiếng Anh là hiện tượng **từ đa âm tiết** (ví dụ: "học sinh" là một từ chứ không phải hai từ rời rạc). Nếu không xử lý tốt bước này, mô hình sẽ hiểu sai ngữ nghĩa hoàn toàn.

### 1. Chuẩn hóa văn bản (Normalization)
Đây là bước "tẩy trần" cho dữ liệu thô.
* **Unicode chuẩn:** Đưa tất cả về dạng Unicode dựng sẵn để tránh lỗi chữ "hòa" và "hoà" (nhìn giống nhau nhưng mã hóa khác nhau).
* **Xử lý dấu câu & ký tự đặc biệt:** Loại bỏ các ký tự không đóng góp vào ý nghĩa chủ đề như `@, #, $, %, ...`.
* **Chuyển chữ thường (Lowercasing):** Giúp giảm kích thước từ điển (Vocabulary size).

### 2. Tách từ (Word Segmentation)


Đây là "đặc sản" của xử lý ngôn ngữ tự nhiên tiếng Việt.
* **Công cụ khuyên dùng:** * **VnCoreNLP:** Độ chính xác rất cao, phù hợp cho các bài toán cần sự khắt khe.
    * **Underthesea:** Dễ cài đặt, tốc độ nhanh, phù hợp cho các dự án linh hoạt.
* **Kết quả:** Cụm từ "Thành phố Hồ Chí Minh" sẽ được chuyển thành `Thành_phố_Hồ_Chí_Minh` để mô hình hiểu đây là một thực thể duy nhất.

### 3. Loại bỏ từ dừng (Stopwords)
Trong phân loại chủ đề, những từ xuất hiện quá nhiều nhưng không mang nội dung đặc trưng sẽ làm nhiễu mô hình.
* **Danh sách Stopwords:** Các từ như `những`, `các`, `của`, `và`, `là`, `đã`...
* **Lưu ý:** Đừng lạm dụng việc xóa stopword nếu bạn dùng mô hình Transformer (như PhoBERT), vì các mô hình này cần ngữ cảnh của cả câu để hiểu nghĩa.

### 4. Vector hóa (Vectorization) - Chuyển chữ thành số
Máy tính không đọc được chữ, nó chỉ làm việc với con số.
* **TF-IDF:** Tốt cho các mô hình Machine Learning truyền thống (SVM, Random Forest). Nó nhấn mạnh vào các từ "đắt" (xuất hiện nhiều trong bài đó nhưng ít trong các bài khác).
* **Word Embeddings (PhoBERT):** Đây là đỉnh cao cho tiếng Việt hiện nay. Nó chuyển từ ngữ thành các vector không gian, giúp mô hình hiểu được "bóng đá" và "cầu thủ" có liên quan mật thiết đến nhau dù mặt chữ khác nhau.

---

## Bảng tóm tắt quy trình xử lý

| Bước | Hành động cụ thể | Công cụ/Thư viện |
| :--- | :--- | :--- |
| **Clean** | Xóa HTML, xóa link, chuẩn hóa dấu | `re` (Regex), `unicodedata` |
| **Tokenize** | Tách từ (Word Segmentation) | `underthesea`, `VnCoreNLP` |
| **Filter** | Loại bỏ Stopwords, ký tự đặc biệt | Bộ từ điển Stopwords tiếng Việt |
| **Embed** | Chuyển văn bản sang vector | `TfidfVectorizer` hoặc `PhoBERT` |

## Data - Folder config

In [13]:
item = {
    "metadata": {
        "doc_id": "5059392",
        "source_name": "VnExpress",
        "source_url": "https://vnexpress.net/sinner-uoc-doi-duoc-danh-hieu-lay-ve-du-world-cup-cho-italy-5059392.html",
        "publish_date": "Thứ ba, 7/4/2026, 10:43 (GMT+7)",
        "author": "VnExpress",
    },
    "content": {
        "title": "\nSinner ước đổi được danh hiệu lấy vé dự World Cup cho Italy ",
        "sapo": "Tay vợt số hai thế giới Jannik Sinner sẵn sàng từ bỏ một trong những chức vô địch ATP để đội tuyển bóng đá Italy được dự World Cup 2026.",
        "content": '"Tôi rất sẵn lòng đánh đổi. Nhiều người trẻ ở Italy chưa từng được chứng kiến đội tuyển quốc gia thi đấu ở World Cup", Sinner trả lời câu hỏi của một phóng viên  hôm 6/4, khi dự giải Monte Carlo Masters tuần này. Thua Bosnia&Herzegovina ở   World Cup hôm 1/4, đội tuyển bóng đá Italy lần thứ ba liên tiếp vắng mặt ở ngày hội bóng đá lớn nhất hành tinh - sân chơi mà họ từng 4 lần vô địch.',
        "word_count": 82,
    },
    "labeling": {
        "original_category": ["Thể thao", "Tennis"],
        "target_label": "Thể thao",
        "tags": ["Pháp, Jannik Sinner, World Cup, World Cup 2026, bóng đá Italy"],
        "is_multilabel": False,
    },
}

## Chuẩn hóa văn bản (Normalization)

In [14]:
import re
import unicodedata


def normalize_text(text):
    """
    Hàm pipeline tổng hợp để làm sạch và chuẩn hóa văn bản.
    """
    if not isinstance(text, str):
        return ""

    # Chuẩn hóa Unicode (NFC)
    # Đưa về dạng Dựng sẵn để tránh lỗi mã hóa "hòa" (1 ký tự) vs "hòa" (nhiều ký tự ghép)
    text = unicodedata.normalize("NFC", text)

    text = text.lower()

    # Loại bỏ rác (Noise Removal) bằng Regex
    text = re.sub(r"<.*?>", " ", text)  # Xóa các thẻ HTML (nếu còn sót)
    text = re.sub(
        r"http\S+|www\S+|https\S+", " ", text, flags=re.MULTILINE
    )  # Xóa Links/URLs
    text = re.sub(r"\S+@\S+", " ", text)  # Xóa địa chỉ Email

    # Xóa ký tự đặc biệt và dấu câu (Chỉ giữ lại chữ cái và số)
    # Lưu ý: Nếu bạn dùng các mô hình cần dấu chấm/phẩy để hiểu ngữ cảnh câu (như PhoBERT),
    # hãy cân nhắc KHÔNG chạy dòng regex này, hoặc chỉ xóa các ký tự biểu tượng (#, @, *, ...).
    text = re.sub(r"[^\w\s]", " ", text)

    # Chuẩn hóa khoảng trắng
    # Biến nhiều dấu cách liên tiếp, tab, hoặc xuống dòng thành 1 dấu cách duy nhất
    text = re.sub(r"\s+", " ", text).strip()

    return text

## Tách từ (Word Segmentation) - Loại bỏ stopwords

In [15]:
from underthesea import word_tokenize


def segment_vietnamese_text(text):
    """
    Hàm tách từ tiếng Việt.
    Đầu vào: Chuỗi văn bản đã được chuẩn hóa (Cleaned text).
    Đầu ra: Chuỗi văn bản đã được ghép từ (ví dụ: 'học_sinh').
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    # Sử dụng hàm word_tokenize với định dạng 'text' để tự động thêm dấu '_'
    segmented_text = word_tokenize(text, format="text")

    return segmented_text

In [16]:
def load_stopwords(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        # Đọc từng dòng, xóa khoảng trắng thừa và đưa vào Set
        return set([line.strip() for line in f])

In [17]:
def remove_stopwords(segmented_text: str, stopwords_set):
    """
    Hàm loại bỏ từ dừng.
    Đầu vào: Chuỗi văn bản đã được TÁCH TỪ (bằng underthesea).
    """
    if not isinstance(segmented_text, str) or not segmented_text.strip():
        return ""

    # Tách chuỗi thành mảng các từ dựa trên khoảng trắng
    words = segmented_text.split()

    # Lọc giữ lại những từ KHÔNG nằm trong danh sách stopword
    filtered_words = [word for word in words if word not in stopwords_set]

    # Ghép mảng trở lại thành chuỗi
    return " ".join(filtered_words)

## Run

In [18]:
stopwords_dash = load_stopwords("./vietnamese-stopwords-dash.txt")
print(len(stopwords_dash))

1942


In [19]:
text: str = item.get("content").get("content")
word_count = item.get("content").get("word_count")
normalized_text = normalize_text(text)

segmented_text = segment_vietnamese_text(normalized_text)
cleaned_text = remove_stopwords(segmented_text, stopwords_dash)

print("Word count origin: ", word_count)
print("Word count aftere preprocessing", len(cleaned_text.split()))

Word count origin:  82
Word count aftere preprocessing 40


In [20]:
print("Original text: ", text)
print("Cleaned text: ", cleaned_text)

Original text:  "Tôi rất sẵn lòng đánh đổi. Nhiều người trẻ ở Italy chưa từng được chứng kiến đội tuyển quốc gia thi đấu ở World Cup", Sinner trả lời câu hỏi của một phóng viên  hôm 6/4, khi dự giải Monte Carlo Masters tuần này. Thua Bosnia&Herzegovina ở   World Cup hôm 1/4, đội tuyển bóng đá Italy lần thứ ba liên tiếp vắng mặt ở ngày hội bóng đá lớn nhất hành tinh - sân chơi mà họ từng 4 lần vô địch.
Cleaned text:  sẵn_lòng đánh_đổi trẻ italy chứng_kiến đội_tuyển quốc_gia thi_đấu world_cup sinner trả_lời câu phóng_viên hôm 6 4 dự_giải monte carlo masters tuần thua bosnia herzegovina world cup hôm 1 4 đội_tuyển bóng_đá italy liên_tiếp vắng_mặt hội bóng_đá hành_tinh sân_chơi 4 vô_địch


text_cleaned, label, original_url

In [21]:
def preprocess_text(text, stopwords_set):
    normalized = normalize_text(text)
    segmented = segment_vietnamese_text(normalized)
    cleaned = remove_stopwords(segmented, stopwords_set)
    return cleaned

In [22]:
import os
import json
from pathlib import Path

def open_data_files(raw_data_path):
    data_files: list[str] = os.listdir(raw_data_path)
    print("Các file dữ liệu có trong thư mục:", data_files)

    data = []
    for file in data_files:
        with open(os.path.join(raw_data_path, file), "r", encoding="utf-8") as f:
            # Dùng List Comprehension để code ngắn gọn và chạy nhanh hơn
            data.extend([json.loads(line) for line in f])
    return data


def save_data_to_folder(
    data_list, folder_name="data/processed", file_name="news_dataset.jsonl"
):
    # 1. Tạo đường dẫn thư mục (Tự động tạo thư mục cha nếu chưa có)
    output_dir = Path(folder_name)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 2. Tạo đường dẫn file đầy đủ
    file_path = output_dir / file_name

    # 3. Ghi dữ liệu vào file theo định dạng JSON Lines
    with open(file_path, "w", encoding="utf-8") as f:
        for item in data_list:
            # Chuyển dict thành string và ghi vào 1 dòng
            line = json.dumps(item, ensure_ascii=False)
            f.write(line + "\n")

    print(f"Đã lưu thành công {len(data_list)} bài báo vào: {file_path}")

In [23]:
def save_data_to_folder_with_txt(
    data_list: list[str], folder_name="", file_name=""
):
    if not folder_name or not file_name:
        print("Vui lòng cung cấp tên thư mục và tên file hợp lệ.")
        return
    # 1. Tạo đường dẫn thư mục (Tự động tạo thư mục cha nếu chưa có)
    output_dir = Path(folder_name)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 2. Tạo đường dẫn file đầy đủ
    file_path = output_dir / file_name

    # 3. Ghi dữ liệu vào file theo định dạng JSON Lines
    with open(file_path, "w", encoding="utf-8") as f:
        for item in data_list:
            f.write(item + "\n")

    print(f"Đã lưu thành công {len(data_list)} bài báo vào: {file_path}")

In [24]:
import datetime
data_raw = open_data_files("../bai-tap-nhom/data")

preprocessed_data: list[str] = []
for item in data_raw:
    original_content = item.get("content", {}).get("content", "")
    cleaned_content = preprocess_text(original_content, stopwords_dash)
    preprocessed_data.append(cleaned_content)

timestamp = datetime.datetime.now().timestamp()
save_data_to_folder_with_txt(preprocessed_data, folder_name="../bai-tap-nhom/processed-data", 
                    file_name=f"data_{int(timestamp)}.txt")

Các file dữ liệu có trong thư mục: ['news_data_1776779386_part1_spider_spider50_1.jsonl']
Đã lưu thành công 150 bài báo vào: ..\bai-tap-nhom\processed-data\data_1776785457.txt
